In [0]:
#The first step is to create the SparkSession object, in order to use Spark.
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName('structured_streaming').getOrCreate()

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *

In [0]:
display(spark.sql("SHOW CATALOGS"))

catalog
my_catalog
samples
system
workspace


In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS my_catalog")
spark.sql("CREATE SCHEMA IF NOT EXISTS my_catalog.my_schema")
spark.sql("""
    CREATE VOLUME IF NOT EXISTS my_catalog.my_schema.my_volume
    COMMENT 'Volume for streaming CSV data'
""")

DataFrame[]

In [0]:
#We create some self-generated data that can be pushed into a
#respective local directory ("csv folder"), to be read by the Structured Streaming. 
#You may need to change the path of the destination file to volumes/my_catalog/my_schema/my_volume/csv_folder
df_1 = spark.createDataFrame(
    [
        ("XN203", "FB", 300, 30),
        ("XN201", "Twitter", 10, 19),
        ("XN202", "Insta", 500, 45)
    ],
    [
        "user_id",
        "app",
        "time_in_secs",
        "age"
    ]
).write.csv(
    "/Volumes/my_catalog/my_schema/my_volume/csv_folder",
    mode="append"
)

In [0]:
schema=StructType().add("user_id","string").add("app","string").add("time_in_secs", "integer").add("age","integer")

In [0]:
#Now that we have one file available in the local folder ("csv folder"),we can go ahead and read it as a stream dataframe. The API to read a #static dataframe is similar to that for reading a streaming dataframe, the only difference being that we use readStream.
data=spark.readStream.option("sep", ",").schema(schema).csv("/Volumes/my_catalog/my_schema/my_volume/csv_folder")

In [0]:
#To validate the schema of the dataframe, we can use the printSchema command.
data.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- app: string (nullable = true)
 |-- time_in_secs: integer (nullable = true)
 |-- age: integer (nullable = true)



In [0]:
app_count=data.groupBy('app').count()

In [0]:
query = (
    app_count.writeStream
    .queryName('count_query')
    .outputMode('complete')
    .format('memory')
    .option(
        "checkpointLocation",
        "/Volumes/my_catalog/my_schema/my_volume/checkpoints/count_query"
    )
    .trigger(availableNow=True)
    .start()
)

In [0]:
spark.sql("select * from count_query ").toPandas().head(5)

,app,count
0,Insta,12
1,FB,24
2,Twitter,12


In [0]:
#In this example, a query is being written to filter only the records of the
#Facebook (FB) app. The average time spent by each user on the FB app is then calculated
fb_data=data.filter(data['app']=='FB')

In [0]:
fb_avg_time=fb_data.groupBy('user_id').agg(F.avg("time_in_secs"))

In [0]:

fb_query = (
    fb_avg_time.writeStream
    .queryName('fb_query')
    .outputMode('complete')
    .format('memory')
    .option(
        "checkpointLocation",
        "/Volumes/my_catalog/my_schema/my_volume/checkpoints/fb_query"
    )
    .trigger(availableNow=True)
    .start()
)

In [0]:
spark.sql("select * from fb_query ").toPandas().head(5)

,user_id,avg(time_in_secs)
0,XN203,350.0
1,XN201,10.0
2,XN202,2000.0


In [0]:
#Because there is only one dataframe currently in the local folder, we get the output of one user accessing FB and the time spent. In order to #view more relative results, let’s push more self-generated data to the folder.

In [0]:
df_2 = spark.createDataFrame(
    [
        ("XN203", "FB", 100, 30),
        ("XN201", "FB", 10, 19),
        ("XN202", "FB", 2000, 45)
    ],
    [
        "user_id",
        "app",
        "time_in_secs",
        "age"
    ]
).write.csv(
    "/Volumes/my_catalog/my_schema/my_volume/csv_folder",
    mode="append"
)

In [0]:
spark.sql("select * from fb_query ").toPandas().head(5)

,user_id,avg(time_in_secs)
0,XN203,350.0
1,XN201,10.0
2,XN202,2000.0


In [0]:
#Now, we have the average time spent across all users using the FB app.
#Let’s add few more records to the folder
df_3=spark.createDataFrame([("XN203",'FB',500,30),
("XN201",'Insta',30,19),("XN202",'Twitter',100,45)],
["user_id","app","time_in_secs","age"]).write.csv("/Volumes/my_catalog/my_schema/my_volume/csv_folder",
mode='append')

In [0]:
spark.sql("select * from fb_query ").toPandas().head(5)

,user_id,avg(time_in_secs)
0,XN203,350.0
1,XN201,10.0
2,XN202,2000.0


In [0]:
#In this example, we see aggregation and sorting of the query on the existing dataframe in the local folder. We group all the records by app #and calculate the total time spent on each app, in decreasing order.
app_df=data.groupBy('app').agg(F.sum('time_in_secs').alias('total_time')).orderBy('total_time',ascending=False)

In [0]:
app_query = (
    app_df.writeStream
    .queryName('app_wise_query')
    .outputMode('complete')
    .format('memory')
    .option(
        "checkpointLocation",
        "/Volumes/my_catalog/my_schema/my_volume/checkpoints/app_wise_query"
    )
    .trigger(availableNow=True)
    .start()
)

In [0]:
spark.sql("select * from app_wise_query ").toPandas().head(5)
#We now have the results for each app and the total time spent by all users on the respective app, using a stream dataframe.

,app,total_time
0,FB,12840
1,Insta,1710
2,Twitter,730


In [0]:
df_4 = spark.createDataFrame(
    [
        ("XN203", "FB", 500, 30),
        ("XN201", "Insta", 30, 19),
        ("XN202", "Twitter", 100, 45)
    ],
    [
        "user_id",
        "app",
        "time_in_secs",
        "age"
    ]
).write.csv(
    "/Volumes/my_catalog/my_schema/my_volume/csv_folder",
    mode="append"
)

In [0]:
spark.sql("select * from app_wise_query ").toPandas().head(5)

,app,total_time
0,FB,12840
1,Insta,1710
2,Twitter,730


In [0]:
#In this example, we try to find the average age of users for every app in
#our data. We simply group the data by app, take the average age of all the
#users, and sort the results in decreasing order.
age_df=data.groupBy('app').agg(F.avg('age').alias('mean_age')).orderBy('mean_age',ascending=False)

In [0]:
age_query = (
    age_df.writeStream
    .queryName('age_query')
    .outputMode('complete')
    .format('memory')
    .option(
        "checkpointLocation",
        "/Volumes/my_catalog/my_schema/my_volume/checkpoints/age_query"
    )
    .trigger(availableNow=True)
    .start()
)

In [0]:
df_5=spark.createDataFrame([("XN210",'FB',500,50),
("XN255",'Insta',30,23),("XN222",'Twitter',100,30)],
["user_id","app","time_in_secs","age"]).write.csv("/Volumes/my_catalog/my_schema/my_volume/checkpoints/age_query",mode='append')

In [0]:
spark.sql("select * from age_query ").toPandas().head(5)

,app,mean_age
0,Twitter,37.909091
1,FB,30.695652
2,Insta,26.090909


In [0]:
app_df=spark.createDataFrame([('FB','FACEBOOK'),('Insta',
'INSTAGRAM'),('Twitter','TWITTER')],["app", "full_name"])
app_df.show()


+-------+---------+
|    app|full_name|
+-------+---------+
|     FB| FACEBOOK|
|  Insta|INSTAGRAM|
|Twitter|  TWITTER|
+-------+---------+



In [0]:
app_stream_df = data.join(app_df, 'app')

join_query = (
    app_stream_df.writeStream
    .queryName('join_query')
    .outputMode('append')
    .format('memory')
    .option(
        "checkpointLocation",
        "/Volumes/my_catalog/my_schema/my_volume/checkpoints/join_query/offsets"
    )
    .trigger(availableNow=True)
    .start()
)

In [0]:
spark.sql("select * from join_query ").toPandas().head(50)

,app,user_id,time_in_secs,age,full_name
0,Twitter,XN202,100,45,TWITTER
1,Twitter,XN202,100,45,TWITTER
2,Twitter,XN202,100,45,TWITTER
3,Twitter,XN202,100,45,TWITTER
4,Twitter,XN202,100,45,TWITTER
5,Twitter,XN202,100,45,TWITTER
6,Twitter,XN202,100,45,TWITTER
7,Twitter,XN202,100,45,TWITTER
8,Twitter,XN202,100,45,TWITTER
9,Twitter,XN202,100,45,TWITTER
